# Silver Layer — Data Cleansing and Standardization

**Notebook:** `02_silver_transformation`  
**Layer:** Silver (Cleaned / Conformed)  
**Lakehouse:** `RetailSalesLakehouse`  
**DP-700 Skill:** Ingest and Transform Data > Design Loading Patterns

## Purpose
Transforms Bronze raw data into Silver by applying:
- **Type casting**: String → correct types (dates, decimals, booleans)
- **Data cleansing**: Trimming whitespace, standardizing case, null handling
- **Deduplication**: Remove exact duplicate rows
- **Validation**: Flag records that fail business rules (invalid emails, negative prices, etc.)
- **SCD Type 1 Upserts**: Merge existing Silver data with new Bronze records using Delta MERGE
- **Schema enforcement**: Explicit schema applied at write time

In [ ]:
# Pipeline Parameters
load_mode = "full"  # 'full' | 'incremental'
watermark_date = "1900-01-01"
run_id = "manual"

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, trim, upper, lower, regexp_replace,
    to_date, to_timestamp, when, coalesce, current_timestamp,
    count, sum as spark_sum, row_number, desc
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, BooleanType, DateType, DecimalType
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

print(f"Silver transformation started. Load mode: {load_mode}")

## 1. Silver: Customers

In [ ]:
df_customers_bronze = spark.table(f"{BRONZE_SCHEMA}.customers")

# Apply incremental filter
if load_mode == "incremental":
    df_customers_bronze = df_customers_bronze.filter(
        col("registration_date") > watermark_date
    )

# Deduplicate — keep the most recently ingested record per customer_id
window_spec = Window.partitionBy("customer_id").orderBy(desc("_ingestion_timestamp"))
df_customers_dedup = (
    df_customers_bronze
    .withColumn("_row_rank", row_number().over(window_spec))
    .filter(col("_row_rank") == 1)
    .drop("_row_rank")
)

# Apply transformations
df_customers_silver = (
    df_customers_dedup
    # Type casting
    .withColumn("date_of_birth", to_date(col("date_of_birth"), "yyyy-MM-dd"))
    .withColumn("registration_date", to_date(col("registration_date"), "yyyy-MM-dd"))
    .withColumn("is_active", when(lower(col("is_active")) == "true", True).otherwise(False))
    # Standardization
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("gender", upper(trim(col("gender"))))
    .withColumn("segment", trim(col("segment")))
    .withColumn("country", upper(trim(col("country"))))
    # Derived columns
    .withColumn("full_name", trim(col("first_name") + lit(" ") + col("last_name")))
    # Data quality flags
    .withColumn("_dq_email_valid", col("email").rlike(r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$"))
    .withColumn("_dq_dob_valid", col("date_of_birth").isNotNull())
    # Silver metadata
    .withColumn("_silver_updated_at", current_timestamp())
    .withColumn("_pipeline_run_id", lit(run_id))
    # Drop Bronze audit cols
    .drop("_source_table", "_source_file", "_load_mode")
)

print(f"Customers after cleansing: {df_customers_silver.count():,} rows")
invalid_emails = df_customers_silver.filter(~col("_dq_email_valid")).count()
print(f"  Invalid emails flagged: {invalid_emails}")

In [ ]:
# SCD Type 1 Upsert using Delta MERGE
# On incremental loads: update existing records, insert new ones
# On full loads: overwrite

if load_mode == "full":
    df_customers_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_SCHEMA}.customers")
    print(f"[FULL LOAD] silver.customers written: {df_customers_silver.count():,} rows")
else:
    # Check if target table exists
    try:
        target = DeltaTable.forName(spark, f"{SILVER_SCHEMA}.customers")
        (
            target.alias("target")
            .merge(
                df_customers_silver.alias("source"),
                "target.customer_id = source.customer_id"
            )
            .whenMatchedUpdateAll()     # SCD Type 1: overwrite existing
            .whenNotMatchedInsertAll()  # New records
            .execute()
        )
        print(f"[INCREMENTAL] silver.customers MERGE completed")
    except Exception:
        df_customers_silver.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.customers")
        print(f"[INCREMENTAL] silver.customers created (first run)")

## 2. Silver: Products

In [ ]:
df_products_bronze = spark.table(f"{BRONZE_SCHEMA}.products")

df_products_silver = (
    df_products_bronze
    # Type casting
    .withColumn("unit_price", col("unit_price").cast(DecimalType(10, 2)))
    .withColumn("cost_price", col("cost_price").cast(DecimalType(10, 2)))
    .withColumn("stock_quantity", col("stock_quantity").cast(IntegerType()))
    .withColumn("weight_kg", col("weight_kg").cast(DoubleType()))
    .withColumn("is_active", when(lower(col("is_active")) == "true", True).otherwise(False))
    .withColumn("created_date", to_date(col("created_date"), "yyyy-MM-dd"))
    # Standardization
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("subcategory", trim(col("subcategory")))
    .withColumn("brand", trim(col("brand")))
    # Derived: margin calculation
    .withColumn("margin_pct",
        when(col("unit_price") > 0,
            ((col("unit_price") - col("cost_price")) / col("unit_price") * 100).cast(DecimalType(5, 2))
        ).otherwise(lit(None))
    )
    # Data quality
    .withColumn("_dq_price_valid", col("unit_price") > 0)
    .withColumn("_dq_cost_less_than_price", col("cost_price") < col("unit_price"))
    # Metadata
    .withColumn("_silver_updated_at", current_timestamp())
    .drop("_source_table", "_source_file", "_load_mode")
)

df_products_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_SCHEMA}.products")
print(f"silver.products: {df_products_silver.count():,} rows")

## 3. Silver: Orders

In [ ]:
df_orders_bronze = spark.table(f"{BRONZE_SCHEMA}.orders")

if load_mode == "incremental":
    df_orders_bronze = df_orders_bronze.filter(col("order_date") > watermark_date)

df_orders_silver = (
    df_orders_bronze
    # Type casting
    .withColumn("order_date", to_date(trim(col("order_date")), "yyyy-MM-dd"))
    .withColumn("ship_date", to_date(trim(col("ship_date")), "yyyy-MM-dd"))
    .withColumn("delivery_date", to_date(trim(col("delivery_date")), "yyyy-MM-dd"))
    .withColumn("shipping_cost", col("shipping_cost").cast(DecimalType(10, 2)))
    .withColumn("discount_amount", col("discount_amount").cast(DecimalType(10, 2)))
    .withColumn("subtotal", col("subtotal").cast(DecimalType(12, 2)))
    .withColumn("tax_amount", col("tax_amount").cast(DecimalType(10, 2)))
    .withColumn("total_amount", col("total_amount").cast(DecimalType(12, 2)))
    # Standardization
    .withColumn("status", trim(col("status")))
    .withColumn("payment_method", trim(col("payment_method")))
    .withColumn("shipping_method", trim(col("shipping_method")))
    .withColumn("channel", trim(col("channel")))
    # Nullify empty coupon codes
    .withColumn("coupon_code",
        when(trim(col("coupon_code")) == "", None).otherwise(trim(col("coupon_code")))
    )
    # Derived columns
    .withColumn("days_to_ship",
        when(col("ship_date").isNotNull() & col("order_date").isNotNull(),
            (col("ship_date").cast("long") - col("order_date").cast("long")) / 86400
        ).cast(IntegerType())
    )
    .withColumn("days_to_deliver",
        when(col("delivery_date").isNotNull() & col("order_date").isNotNull(),
            (col("delivery_date").cast("long") - col("order_date").cast("long")) / 86400
        ).cast(IntegerType())
    )
    # Data quality
    .withColumn("_dq_total_positive", col("total_amount") >= 0)
    .withColumn("_dq_order_date_not_null", col("order_date").isNotNull())
    # Metadata
    .withColumn("_silver_updated_at", current_timestamp())
    .drop("_source_table", "_source_file", "_load_mode")
)

if load_mode == "full":
    df_orders_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_SCHEMA}.orders")
else:
    try:
        target = DeltaTable.forName(spark, f"{SILVER_SCHEMA}.orders")
        (target.alias("t").merge(df_orders_silver.alias("s"), "t.order_id = s.order_id")
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    except Exception:
        df_orders_silver.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.orders")

print(f"silver.orders: {spark.table(f'{SILVER_SCHEMA}.orders').count():,} rows")

## 4. Silver: Order Items

In [ ]:
df_items_bronze = spark.table(f"{BRONZE_SCHEMA}.order_items")

df_items_silver = (
    df_items_bronze
    .withColumn("quantity", col("quantity").cast(IntegerType()))
    .withColumn("unit_price", col("unit_price").cast(DecimalType(10, 2)))
    .withColumn("discount_pct", col("discount_pct").cast(DecimalType(5, 2)))
    .withColumn("line_total", col("line_total").cast(DecimalType(12, 2)))
    # Validate
    .withColumn("_dq_quantity_positive", col("quantity") > 0)
    .withColumn("_dq_line_total_valid",
        (col("line_total") >= 0) &
        (col("line_total") <= col("unit_price") * col("quantity"))
    )
    .withColumn("_silver_updated_at", current_timestamp())
    .drop("_source_table", "_source_file", "_load_mode")
)

write_mode = "append" if load_mode == "incremental" else "overwrite"
df_items_silver.write.format("delta").mode(write_mode).option("overwriteSchema", "true").saveAsTable(f"{SILVER_SCHEMA}.order_items")
print(f"silver.order_items: {df_items_silver.count():,} rows")

## 5. Silver Layer Summary

In [ ]:
print("=" * 60)
print("SILVER LAYER TRANSFORMATION SUMMARY")
print("=" * 60)
for table in ["customers", "products", "orders", "order_items"]:
    count = spark.table(f"{SILVER_SCHEMA}.{table}").count()
    print(f"  silver.{table:20s}: {count:>10,} rows")

# Data quality report
print("\nDATA QUALITY REPORT")
print("-" * 40)
dq_customers = spark.table(f"{SILVER_SCHEMA}.customers")
total = dq_customers.count()
valid_emails = dq_customers.filter(col("_dq_email_valid")).count()
print(f"  Customers - Valid emails: {valid_emails}/{total} ({valid_emails/total*100:.1f}%)")